In [ ]:
import os
import glob
import nibabel as nib
import numpy as np
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
from scipy.io import savemat
from lameg.util import load_meg_sensor_data, big_brain_proportional_layer_boundaries, convert_native_to_fsaverage


In [ ]:
# Set FreeSurfer environment variables
os.environ['FREESURFER_HOME'] = '/usr/local/freesurfer'
os.environ['FSFAST_HOME'] = '/usr/local/freesurfer/fsfast'
os.environ['SUBJECTS_DIR'] = '/home/membercophy/Matteo_M/V1_ERF/BrainDyn_Data/Data/MRI'

# Add FreeSurfer binaries to PATH
os.environ['PATH'] = (
    f"{os.environ['FREESURFER_HOME']}/bin:"
    f"{os.environ['FREESURFER_HOME']}/fsfast/bin:"
    f"{os.environ['FREESURFER_HOME']}/tktools:"
    f"{os.environ['FREESURFER_HOME']}/mni/bin:"
    f"{os.environ['PATH']}"
)

# Print for verification
print(f"FREESURFER_HOME: {os.environ['FREESURFER_HOME']}")
print(f"FSFAST_HOME: {os.environ['FSFAST_HOME']}")
print(f"SUBJECTS_DIR: {os.environ['SUBJECTS_DIR']}")
print(f"PATH: {os.environ['PATH']}")

In [ ]:

def align_to_abs_peak_with_edges_time(
    X,
    start_time=-0.3,     # start of the data
    peak_window=(0.05, 0.150),  # where to find a peak
    fs=600
):
    """
    X : array (n_series, n_time)
    start_time : time of first sample (-0.3 или -0.5)
    peak_window : (t1, t2) in seconds — the window within which we search for |peak|
    fs : sampling frequency (default 600 Hz)

    Returns:
    lags — shifts (n_series,)
    align_idx — indexes for alignment (list)
    peaks — peaks found for each participant
    ref_peak — average peak position
    """

    X = np.asarray(X)
    n_series, n_time = X.shape
    times = start_time + np.arange(n_time) / fs
    t1, t2 = peak_window
    i1 = np.searchsorted(times, t1)
    i2 = np.searchsorted(times, t2)
    i1 = max(0, i1)
    i2 = min(n_time, i2)
    peaks = np.array([
        i1 + np.argmax(np.abs(x[i1:i2]))
        for x in X
    ])
    ref_peak = int(round(np.mean(peaks)))
    lags = ref_peak - peaks

    base = np.arange(n_time)
    align_idx = [
        np.clip(base - lag, 0, n_time - 1)
        for lag in lags
    ]

    return lags, align_idx, peaks, ref_peak

def split_into_6_layers(n_layers, bb_bounds):
    layers = np.arange(n_layers)
    idx = []

    raw_idx = [
        np.where(layers < bb_bounds[0])[0],
        np.where((layers >= bb_bounds[0]) & (layers < bb_bounds[1]))[0],
        np.where((layers >= bb_bounds[1]) & (layers < bb_bounds[2]))[0],
        np.where((layers >= bb_bounds[2]) & (layers < bb_bounds[3]))[0],
        np.where((layers >= bb_bounds[3]) & (layers < bb_bounds[4]))[0],
        np.where(layers >= bb_bounds[4])[0],
    ]
    
    for i in range(6):

        if len(raw_idx[i]) > 0:
            idx.append(raw_idx[i])
            continue

        left = i - 1
        while left >= 0 and len(raw_idx[left]) == 0:
            left -= 1
        right = i + 1
        while right <= 5 and len(raw_idx[right]) == 0:
            right += 1
        if left >= 0 and right <= 5:
            merged = np.arange(raw_idx[left][-1], raw_idx[right][0] + 1)
            idx.append(merged)
            print('merged')    
    return idx

def minmax_unit(x):
    x = np.asarray(x)
    xmin, xmax = np.min(x), np.max(x)
    if xmax == xmin:
        return np.zeros_like(x)
    return 2 * (x - xmin) / (xmax - xmin) - 1


def get_bigbrain_layer_boundaries(subj_id, subj_surf_dir, subj_coord=None):
    """
    Get the cortical layer boundaries based on the Big Brain atlas for a specified coordinate
    in the subject's downsampled combined space. If subj_coord is None, this function returns
    the 6 proportional layer boundaries for every vertex in the downsampled mesh.

    Parameters
    ----------
    subj_id : str
        The subject identifier for which the conversion is being performed.
    subj_surf_dir : str
        The path containing the laMEG-processed subject surfaces.
    subj_coord : array-like or None, optional
        The x, y, z coordinates on the subject's combined hemisphere pial surface to be
        converted. If None, all downsampled pial vertices are mapped. Defaults to None.

    Returns
    -------
    vert_bb_prop : np.ndarray
        A 6 x M array of proportional layer boundaries (rows = 6 layer boundaries,
        columns = vertices), where M is the number of vertices in subj_coord (if provided)
        or in the downsampled mesh (if subj_coord is None). Values range between 0 and 1,
        which must be scaled by the cortical thickness to get layer depths in millimeters.
    """
    # Convert subject coordinate(s) to fsaverage vertex index
    hemi, fsave_v_idx = convert_native_to_fsaverage(subj_id, subj_surf_dir, subj_coord)

    # Retrieve or compute the Big Brain proportional layer boundaries
    # big_brain_proportional_layer_boundaries() is assumed to return a dict:
    #    {'lh': <6 x N_lh array>, 'rh': <6 x N_rh array>}
    bb_prop = big_brain_proportional_layer_boundaries()

    # If we only have a single coordinate, hemi will be a string; otherwise, it is a list of hemis
    if isinstance(hemi, str):
        # Single coordinate: just index directly
        vert_bb_prop = bb_prop[hemi][:, fsave_v_idx]
    else:
        # Multiple coordinates: build a 6 x M array
        vert_bb_prop = np.zeros((6, len(hemi)))
        for i, (v_h, idx) in enumerate(zip(hemi, fsave_v_idx)):
            vert_bb_prop[:, i] = bb_prop[v_h][:, idx]

    return vert_bb_prop

    
def upload_meg_file(rawdata_path, sub, ses, condition):
    """
    Search and upload the raw datachunck

    :param rawdata_path: datapath
    :param sub: subjects id
    :param ses: session id 
    :param condition: experimental condition (block-wise)
    """
    
    
    sub_meg_path = os.path.join(rawdata_path, "sub-" + sub , "ses-0" + ses , "meg/sub-"+ sub + "_ses-0" + ses + "_task-" + condition + "_acq-")
    folders_acq = glob.glob(sub_meg_path + "*_meg.ds", recursive=True)
    list_folders = sorted(folders_acq, key=lambda x: int(x.split('acq-')[1].split('_')[0]))
    
    return list_folders

In [ ]:
# EXtract the Free Energy 

In [ ]:

subjects=['sub-105', 'sub-107', 'sub-109', 'sub-113', 'sub-114',
           'sub-115', 'sub-117','sub-118','sub-119', 'sub-123', 
           'sub-124', 'sub-126','sub-127','sub-129', 'sub-130',
           'sub-131','sub-132','sub-133','sub-134','sub-135', 
           'sub-136', 'sub-137','sub-138', 'sub-141','sub-142',
           'sub-143','sub-144','sub-145']

sessions = {
    'sub-105': ['ses-01'], 'sub-107': ['ses-01'], 'sub-109': ['ses-01'], 'sub-113': ['ses-01'], 
    'sub-114': ['ses-01'], 'sub-115': ['ses-01'],'sub-117': ['ses-01'], 'sub-118': ['ses-01'], 
    'sub-119': ['ses-01'], 'sub-123': ['ses-01'],'sub-124': ['ses-01'], 'sub-126': ['ses-01'], 
    'sub-127': ['ses-01'], 'sub-129': ['ses-01'],'sub-130': ['ses-01'], 'sub-131': ['ses-01'],
    'sub-132': ['ses-01'], 'sub-133': ['ses-01'],'sub-134': ['ses-01'], 'sub-135': ['ses-01'], 
    'sub-136': ['ses-01'], 'sub-137': ['ses-01'], 'sub-138': ['ses-01'],'sub-141': ['ses-01'], 
    'sub-142': ['ses-01'], 'sub-143': ['ses-01'], 'sub-144': ['ses-01'],
    'sub-145': ['ses-01']
}


n_layers=11
braindyn_files = "/home/membercophy/Matteo_M/V1_ERF/BrainDyn_Data/Data/MRI"
meg_rawdata_path = "/home/membercophy/Matteo_M/V1_ERF/rawdata"
savemat_folder = "/home/membercophy/Matteo_M/V1_ERF/V1_ERF_F_diff"

# all_snrs=[]

SUB = ['105', '107', '109', '113', '114',
        '115', '117','118','119', '123', 
        '124', '126','127','129', '130',
        '131','132','133','134', '135', 
        '136', '137','138', '141','142',
        '143','144','145']

ses = "1"
condition = "V100V100"
mxf_option = "no_mxf" 

snrs_subj = []

condition = "V100V100"

triggers = ["left_stim", "right_stim"]
snr_data = {}
snr_lim = -15
patch_size = ["10", "25", "75"]
window_size = ["25", "50"]

for trigger in triggers:

    snrs_subj = []
    total_trials_sub = []

    for s, sub in enumerate(SUB): 
        time_lims=[50,150]

        list_folders = upload_meg_file(meg_rawdata_path, sub, ses, condition)
        mxf_path_dir = os.path.join(list_folders[-1], f"mxf_{mxf_option}_data")

        output_spm_conv_dir = os.path.join(mxf_path_dir, "mxf_sess_data", "spm_convert_epo")
        spm_converted_file = f"spm_sub-{sub}_ses-0{ses}_task-{condition}_meg_{mxf_option}_{trigger}_nrg_ERF_epo"
        base_fname = os.path.join(output_spm_conv_dir,spm_converted_file + ".mat")

        sensor_data, sensor_time, ch_names = load_meg_sensor_data(base_fname)
        st_idx=np.where((sensor_time>=time_lims[0]) & (sensor_time<=time_lims[1]))[0]
        sensor_data=sensor_data[:,st_idx,:]
        total_trials = sensor_data.shape[2]

        sensor_time=sensor_time[st_idx]
        mean_signal = np.mean(sensor_data, axis=2)
        noise = sensor_data - mean_signal[:, :, np.newaxis]
        rms_signal = np.sqrt(np.mean(mean_signal**2, axis=0))  # shape: (n_times,)
        rms_noise = np.sqrt(np.mean(np.mean(noise**2, axis=2), axis=0))  # shape: (n_times,)
        snr_db = 20 * np.log10(rms_signal / rms_noise)

        snrs_subj.append(snr_db)
        total_trials_sub.append(total_trials)


    snrs_all = np.array(snrs_subj)
    mean_snr_sub = np.mean(snrs_all, axis=1)
    total_trials_sub = np.array(total_trials_sub)

    snr_data[trigger] = {
    'y': np.array(mean_snr_sub),
    'x': np.array(total_trials_sub),
    }

left_idx = np.where(snr_data['left_stim']['y'] < snr_lim)[0]
right_idx = np.where(snr_data['right_stim']['y'] < snr_lim)[0]
lo_snr_subjects = [item for i, item in enumerate(subjects) if i in left_idx or i in right_idx]
hi_snr_subjects = [item for i, item in enumerate(subjects) if i not in left_idx and i not in right_idx]
print(f"LO SNR subj : {lo_snr_subjects}")
print(f" HI SNR subj :{hi_snr_subjects}")

for win in window_size:
    
    for patch in patch_size: 

        #out_dir = f"/home/membercophy/Matteo_M/V1_ERF/results_patch{patch}"
        out_dir = f"/home/membercophy/Matteo_M/V1_ERF/results_time_{patch}"

        for trigger in triggers:

            all_lh_prior_ts=[]
            all_lh_F_diff=[]
            all_lh_layer_F_diff=[]
            all_lh_bb_boundaries=[]

            all_rh_prior_ts=[]
            all_rh_F_diff=[]
            all_rh_layer_F_diff=[]
            all_rh_bb_boundaries=[]

            print(f"Processing {trigger}.")
            for subject in hi_snr_subjects:
                sub_n=subject.split('-')[-1]
                for sess_idx, session in enumerate(sessions[subject]):
                    #out_fname = os.path.join(out_dir,f'results_{subject}_{session}_{trigger}_patch{patch}.npz')
                    out_fname = os.path.join(out_dir,f'results_{subject}_{session}_{trigger}_patch1_ws{win}.npz')

                    if os.path.exists(out_fname):
                        subj_surf_dir=os.path.join(braindyn_files,f'{sub_n}-synth','layer_surf')
                        orig_inflated=nib.load(os.path.join(subj_surf_dir, 'inflated.gii'))
                        ds_inflated=nib.load(os.path.join(subj_surf_dir, 'inflated.ds.gii'))
                        ds_pial = nib.load(os.path.join(subj_surf_dir, 'pial.ds.gii'))

                        data = np.load(out_fname, allow_pickle=True)
                        lh_prior = data['lh_prior']
                        lh_prior_ts = data['lh_prior_ts']
                        rh_prior = data['rh_prior']
                        rh_prior_ts = data['rh_prior_ts']
                        ts_time = data['ts_time']
                        lh_Fs = data['lh_Fs']
                        rh_Fs = data['rh_Fs']
                        wois = data['wois']
                        time_lims=[-300,300]
                        
                        woi_time=np.mean(wois,axis=-1)
                        t_idx=np.where((woi_time>=time_lims[0]) & (woi_time<=time_lims[1]))[0]
                        woi_win=woi_time[t_idx]
                        lh_Fs_win=lh_Fs[:,t_idx]
                        rh_Fs_win=rh_Fs[:,t_idx]

                        ts_idx=np.where((ts_time>=time_lims[0]) & (ts_time<=time_lims[1]))[0]
                        lh_prior_ts=lh_prior_ts[ts_idx]
                        rh_prior_ts=rh_prior_ts[ts_idx]
                        ts_time=ts_time[ts_idx]
                        
                        m_idx=np.argmax(np.abs(lh_prior_ts))
                        if lh_prior_ts[m_idx]>0:
                            lh_prior_ts=-1*lh_prior_ts
                        m_idx=np.argmax(np.abs(rh_prior_ts))
                        if rh_prior_ts[m_idx]>0:
                            rh_prior_ts=-1*rh_prior_ts

                        lh_F_diff = lh_Fs_win.copy()
                        rh_F_diff = rh_Fs_win.copy()
                            
                        lh_surf_bb_bounds=n_layers*get_bigbrain_layer_boundaries(f'{sub_n}-synth', subj_surf_dir, subj_coord=ds_pial.darrays[0].data[lh_prior,:])

                        lh_idx = split_into_6_layers(n_layers, lh_surf_bb_bounds)
                        
                        lh_layer_F = np.vstack([ 
                            np.mean(lh_Fs_win[lh_idx[0], :], axis=0), 
                            np.mean(lh_Fs_win[lh_idx[1], :], axis=0), 
                            np.mean(lh_Fs_win[lh_idx[2], :], axis=0), 
                            np.mean(lh_Fs_win[lh_idx[3], :], axis=0), 
                            np.mean(lh_Fs_win[lh_idx[4], :], axis=0), 
                            np.mean(lh_Fs_win[lh_idx[5], :], axis=0), 
                        ])


                        lh_layer_F_diff = lh_layer_F.copy()
                            
                        rh_surf_bb_bounds=n_layers*get_bigbrain_layer_boundaries(f'{sub_n}-synth', subj_surf_dir, subj_coord=ds_pial.darrays[0].data[rh_prior,:])
                        
                        rh_idx = split_into_6_layers(n_layers, rh_surf_bb_bounds)
                        
                        rh_layer_F = np.vstack([
                            np.mean(rh_Fs_win[rh_idx[0], :], axis=0), 
                            np.mean(rh_Fs_win[rh_idx[1], :], axis=0), 
                            np.mean(rh_Fs_win[rh_idx[2], :], axis=0), 
                            np.mean(rh_Fs_win[rh_idx[3], :], axis=0), 
                            np.mean(rh_Fs_win[rh_idx[4], :], axis=0), 
                            np.mean(rh_Fs_win[rh_idx[5], :], axis=0), 
                        ])
                                        
                        rh_layer_F_diff = rh_layer_F.copy()
                        
                        col_r = plt.cm.cool(np.linspace(0,1, num=n_layers))
                        plt.figure(figsize=(12,8))
                        ax=plt.subplot(2,2,1)
                        ax.plot(ts_time,lh_prior_ts,'k')
                        ax.axvline(x=0, color='k', linestyle='--')
            #             ax2=ax.twinx()
            #             ax2.plot(sensor_time, snr_db)
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel('Amplitude (nAm/mm^2)')
                        ax.set_title(f'{subject}-{session}-LH')
                        ax.set_xlim(time_lims)

                        # Plot relative free energy time series
                        ax=plt.subplot(2,2,2)
                        for l in range(n_layers):
                            if l==0:
                                ax.plot(woi_win, lh_F_diff[l,:], label='superficial', color=col_r[l,:])
                            elif l==n_layers-1:
                                ax.plot(woi_win, lh_F_diff[l,:], label='deep', color=col_r[l,:])
                            else:
                                ax.plot(woi_win, lh_F_diff[l,:], color=col_r[l,:])
                        plt.legend()
                        ax.axvline(x=0, color='k', linestyle='--')
                        ax2=ax.twinx()
                        ax2.plot(ts_time,lh_prior_ts,'k')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel(r'$\Delta$F')
                        ax.set_xlim(time_lims)

                        ax=plt.subplot(2,2,3)
                        h=ax.imshow(lh_F_diff,aspect='auto', extent=[woi_win[0], woi_win[-1], n_layers-1,0],cmap='plasma',origin='upper')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel('Layer')
                        peak_layer=gaussian_filter1d(np.argmax(lh_F_diff,axis=0).astype(float), 2)
                        ax.plot(woi_win, peak_layer, 'w')
                        ax.axvline(x=0, color='k', linestyle='--')
                        for i in range(len(lh_surf_bb_bounds)-1):
                            ax.axhline(y=lh_surf_bb_bounds[i],color='w',linestyle='--')        
                        ax2=ax.twinx()
                        ax2.plot(ts_time,lh_prior_ts,'k')
                        ax2.set_yticks([])

                        ax=plt.subplot(2,2,4)
                        col_l = plt.cm.cool(np.linspace(0,1, num=6))
                        l_labels=['I','II','III','IV','V','VI']
                        for i in range(6):
                            ax.plot(woi_win, lh_layer_F_diff[i,:],label=l_labels[i],color=col_l[i])
                        plt.legend()
                        ax.axvline(x=0, color='k', linestyle='--')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel(r'$\Delta$F')
                        ax2=ax.twinx()
                        ax2.plot(ts_time,lh_prior_ts,'k')
                        ax.set_xlim(time_lims)

                        plt.tight_layout()
                        
                        col_r = plt.cm.cool(np.linspace(0,1, num=n_layers))
                        
                        plt.figure(figsize=(12,8))
                        ax=plt.subplot(2,2,1)
                        ax.plot(ts_time,rh_prior_ts,'k')
                        ax.axvline(x=0, color='k', linestyle='--')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel('Amplitude (nAm/mm^2)')
                        ax.set_title(f'{trigger}{subject}-{session}-RH')
                        ax.set_xlim(time_lims)

                        # Plot free energy time series
                        ax=plt.subplot(2,2,2)
                        for l in range(n_layers):
                            if l==0:
                                ax.plot(woi_win, rh_F_diff[l,:], label='superficial', color=col_r[l,:])
                            elif l==n_layers-1:
                                ax.plot(woi_win, rh_F_diff[l,:], label='deep', color=col_r[l,:])
                            else:
                                ax.plot(woi_win, rh_F_diff[l,:], color=col_r[l,:])
                        plt.legend()
                        ax.axvline(x=0, color='k', linestyle='--')
                        ax2=ax.twinx()
                        ax2.plot(ts_time,rh_prior_ts,'k')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel(r'$\Delta$F')
                        ax.set_xlim(time_lims)

                        ax=plt.subplot(2,2,3)
                        h=ax.imshow(rh_F_diff,aspect='auto', extent=[woi_win[0], woi_win[-1], n_layers-1,0],cmap='plasma',origin='upper')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel('Layer')
                        peak_layer=gaussian_filter1d(np.argmax(rh_F_diff,axis=0).astype(float), 2)
                        ax.plot(woi_win, peak_layer, 'w')
                        ax.axvline(x=0, color='k', linestyle='--')
                        for i in range(len(rh_surf_bb_bounds)-1):
                            ax.axhline(y=rh_surf_bb_bounds[i],color='w',linestyle='--')        
                        ax2=ax.twinx()
                        ax2.plot(ts_time,rh_prior_ts,'k')
                        ax2.set_yticks([])

                        ax=plt.subplot(2,2,4)
                        col_l = plt.cm.cool(np.linspace(0,1, num=6))
                        l_labels=['I','II','III','IV','V','VI']
                        for i in range(6):
                            ax.plot(woi_win, rh_layer_F_diff[i,:],label=l_labels[i],color=col_l[i])
                        plt.legend()
                        ax.axvline(x=0, color='k', linestyle='--')
                        ax.set_xlabel('Time (ms)')
                        ax.set_ylabel(r'$\Delta$F')
                        ax2=ax.twinx()
                        ax2.plot(ts_time,rh_prior_ts,'k')
                        ax.set_xlim(time_lims)

                        plt.tight_layout()
                        
                        plt.show()

                        all_lh_prior_ts.append(lh_prior_ts)
                        all_lh_F_diff.append(lh_F_diff)
                        all_lh_layer_F_diff.append(lh_layer_F_diff)
                        all_lh_bb_boundaries.append(lh_surf_bb_bounds)
                        
                        all_rh_prior_ts.append(rh_prior_ts)
                        all_rh_F_diff.append(rh_F_diff)
                        all_rh_layer_F_diff.append(rh_layer_F_diff)
                        all_rh_bb_boundaries.append(rh_surf_bb_bounds)
                        
                    else:
                        raise FileNotFoundError(f"MISSING FILE: {out_fname} for subject {subject}")

            all_lh_prior_ts=np.array(all_lh_prior_ts)
            all_lh_F_diff=np.array(all_lh_F_diff)
            all_lh_layer_F_diff=np.array(all_lh_layer_F_diff)
            all_lh_bb_boundaries=np.array(all_lh_bb_boundaries)

            all_rh_prior_ts=np.array(all_rh_prior_ts)
            all_rh_F_diff=np.array(all_rh_F_diff)
            all_rh_layer_F_diff=np.array(all_rh_layer_F_diff)
            all_rh_bb_boundaries=np.array(all_rh_bb_boundaries)

            # -------- LH --------
            plt.figure()
            plt.plot(ts_time, minmax_unit(all_lh_prior_ts[0]))
            lh_lags, lh_align_idx, peaks, ref = align_to_abs_peak_with_edges_time(all_lh_prior_ts,
                start_time=-0.3,
                peak_window=(0.05, 0.15))

            aligned_lh_prior_ts = []
            aligned_lh_F_diff = []
            aligned_lh_layer_F_diff = []
            for i in range(all_lh_prior_ts.shape[0]):
                idx = lh_align_idx[i]
                if i > 0:
                    plt.plot(ts_time, minmax_unit(all_lh_prior_ts[i, idx]))
                aligned_lh_prior_ts.append(all_lh_prior_ts[i, idx])
                aligned_lh_F_diff.append(all_lh_F_diff[i, :, idx])
                aligned_lh_layer_F_diff.append(all_lh_layer_F_diff[i, :, idx])

            aligned_lh_prior_ts = np.asarray(aligned_lh_prior_ts)
            aligned_lh_F_diff = np.transpose(np.asarray(aligned_lh_F_diff), (0, 2, 1))
            aligned_lh_layer_F_diff = np.transpose(np.asarray(aligned_lh_layer_F_diff), (0, 2, 1))

            plt.xlabel("Time (ms)")
            plt.ylabel("Amplitude (au)")
            plt.title(f"{trigger} LH")
            plt.tight_layout(); plt.xlim(time_lims); plt.show()

            # -------- RH --------
            plt.figure()
            plt.plot(ts_time, minmax_unit(all_rh_prior_ts[0]))
            rh_lags, rh_align_idx, peaks, ref = align_to_abs_peak_with_edges_time(all_rh_prior_ts,
                start_time=-0.3,
                peak_window=(0.05, 0.15))

            aligned_rh_prior_ts = []
            aligned_rh_F_diff = []
            aligned_rh_layer_F_diff = []
            for i in range(all_rh_prior_ts.shape[0]):
                idx = rh_align_idx[i]
                if i > 0:
                    plt.plot(ts_time, minmax_unit(all_rh_prior_ts[i, idx]))
                aligned_rh_prior_ts.append(all_rh_prior_ts[i, idx])
                aligned_rh_F_diff.append(all_rh_F_diff[i, :, idx])
                aligned_rh_layer_F_diff.append(all_rh_layer_F_diff[i, :, idx])

            aligned_rh_prior_ts = np.asarray(aligned_rh_prior_ts)
            aligned_rh_F_diff = np.transpose(np.asarray(aligned_rh_F_diff), (0, 2, 1))
            aligned_rh_layer_F_diff = np.transpose(np.asarray(aligned_rh_layer_F_diff), (0, 2, 1))

            plt.xlabel("Time (ms)")
            plt.ylabel("Amplitude (au)")
            plt.title(f"{trigger} RH")
            plt.tight_layout(); plt.xlim(time_lims); plt.show()

            if trigger == "right_stim":
                ra_aligned_rh_prior_ts  = aligned_rh_prior_ts
                ra_aligned_lh_prior_ts  = aligned_lh_prior_ts

                ra_aligned_rh_F_diff = aligned_rh_F_diff
                ra_aligned_rh_layer_F_diff = aligned_rh_layer_F_diff

                ra_aligned_lh_F_diff = aligned_lh_F_diff
                ra_aligned_lh_layer_F_diff = aligned_lh_layer_F_diff
                print(f"Saved results for {trigger}")
            elif trigger == "left_stim":
                la_aligned_rh_prior_ts  = aligned_rh_prior_ts
                la_aligned_lh_prior_ts  = aligned_lh_prior_ts

                la_aligned_rh_F_diff = aligned_rh_F_diff
                la_aligned_rh_layer_F_diff = aligned_rh_layer_F_diff

                la_aligned_lh_F_diff = aligned_lh_F_diff
                la_aligned_lh_layer_F_diff = aligned_lh_layer_F_diff
                print(f"Saved results for {trigger}")

            else:
                raise NameError("Trigger not available.")

        # save .mat file the FE
        savemat_fullpath = os.path.join(savemat_folder, f'aligned_V1_ERF_FE_ws{win}_ps{patch}.mat')
        savemat(savemat_fullpath, {
            'la_aligned_rh_prior_ts': la_aligned_rh_prior_ts,
            'la_aligned_lh_prior_ts': la_aligned_lh_prior_ts,

            'la_aligned_rh_F_diff': la_aligned_rh_F_diff,
            'la_aligned_rh_layer_F_diff': la_aligned_rh_layer_F_diff,
            'la_aligned_lh_F_diff': la_aligned_lh_F_diff,
            'la_aligned_lh_layer_F_diff': la_aligned_lh_layer_F_diff,

            'ra_aligned_rh_prior_ts': ra_aligned_rh_prior_ts,
            'ra_aligned_lh_prior_ts': ra_aligned_lh_prior_ts,

            'ra_aligned_rh_F_diff': ra_aligned_rh_F_diff,
            'ra_aligned_rh_layer_F_diff': ra_aligned_rh_layer_F_diff,
            'ra_aligned_lh_F_diff': ra_aligned_lh_F_diff,
            'ra_aligned_lh_layer_F_diff': ra_aligned_lh_layer_F_diff,

            'wois' : wois, 
            'ts_time' : ts_time,
        })

        contra_ts=np.mean(np.stack([la_aligned_rh_prior_ts, ra_aligned_lh_prior_ts]),axis=0)
        ipsi_ts=np.mean(np.stack([la_aligned_lh_prior_ts, ra_aligned_rh_prior_ts]),axis=0)
        contra_ts_1d = np.mean(contra_ts, axis=0)  
        ipsi_ts_1d = np.mean(ipsi_ts, axis=0)     
        plt.plot(ts_time, contra_ts_1d, label='contra')
        plt.plot(ts_time, ipsi_ts_1d, label='ipsi')
        plt.legend()

In [ ]:
# extract the Dfs

In [ ]:

def align_to_abs_peak_with_edges_time(X, start_time=-0.3, peak_window=(0.05, 0.150), fs=600):
    """
    :param X: array (n_series, n_time)
    :param start_time: time of first sample (-0.3 или -0.5)
    :param peak_window; (t1, t2) in seconds — the window within which we search for |peak|
    :param fs: sampling frequency (default 600 Hz)

    Returns:
    lags:  shifts (n_series,)
    align_idx: indexes for alignment (list)
    peaks: peaks found for each participant
    ref_peak: average peak position
    """

    X = np.asarray(X)
    n_series, n_time = X.shape
    times = start_time + np.arange(n_time) / fs
    t1, t2 = peak_window
    i1 = np.searchsorted(times, t1)
    i2 = np.searchsorted(times, t2)
    i1 = max(0, i1)
    i2 = min(n_time, i2)
    peaks = np.array([
        i1 + np.argmax(np.abs(x[i1:i2]))
        for x in X
    ])
    ref_peak = int(round(np.mean(peaks)))
    lags = ref_peak - peaks

    base = np.arange(n_time)
    align_idx = [
        np.clip(base - lag, 0, n_time - 1)
        for lag in lags
    ]

    return lags, align_idx

subjects=['sub-105', 'sub-107', 'sub-109', 'sub-113', 'sub-114',
           'sub-115', 'sub-117','sub-118','sub-119', 'sub-123', 
           'sub-124', 'sub-126','sub-127','sub-129', 'sub-130',
           'sub-131','sub-132','sub-133','sub-134','sub-135', 
           'sub-136', 'sub-137','sub-138', 'sub-141','sub-142',
           'sub-143','sub-144','sub-145']

sessions = {
    'sub-105': ['ses-01'], 'sub-107': ['ses-01'], 'sub-109': ['ses-01'], 'sub-113': ['ses-01'], 
    'sub-114': ['ses-01'], 'sub-115': ['ses-01'],'sub-117': ['ses-01'], 'sub-118': ['ses-01'], 
    'sub-119': ['ses-01'], 'sub-123': ['ses-01'],'sub-124': ['ses-01'], 'sub-126': ['ses-01'], 
    'sub-127': ['ses-01'], 'sub-129': ['ses-01'],'sub-130': ['ses-01'], 'sub-131': ['ses-01'],
    'sub-132': ['ses-01'], 'sub-133': ['ses-01'],'sub-134': ['ses-01'], 'sub-135': ['ses-01'], 
    'sub-136': ['ses-01'], 'sub-137': ['ses-01'], 'sub-138': ['ses-01'],'sub-141': ['ses-01'], 
    'sub-142': ['ses-01'], 'sub-143': ['ses-01'], 'sub-144': ['ses-01'],
    'sub-145': ['ses-01']
}

n_layers=11
#out_dir="/home/membercophy/Matteo_M/V1_ERF/results_patch5"
out_dir ="/home/membercophy/Matteo_M/V1_ERF/ps5_results_winsize"
savemat_folder = "/home/membercophy/Matteo_M/V1_ERF/V1_ERF_F_diff"

braindyn_files = "/home/membercophy/Matteo_M/V1_ERF/BrainDyn_Data/Data/MRI"
meg_rawdata_path = "/home/membercophy/Matteo_M/V1_ERF/rawdata"

# all_snrs=[]


SUB = ['105', '107', '109', '113', '114',
        '115', '117','118','119', '123', 
        '124', '126','127','129', '130',
        '131','132','133','134', '135', 
        '136', '137','138', '141','142',
        '143','144','145']

ses = "1"
condition = "V100V100"
mxf_option = "no_mxf" 
triggers = ["left_stim", "right_stim"]
snr_data = {}
snr_lim = -15
time_lims=[50,150]

for trigger in triggers:
    snrs_subj = []
    total_trials_sub = []

    for s, sub in enumerate(SUB): 
        
        list_folders = upload_meg_file(meg_rawdata_path, sub, ses, condition)
        mxf_path_dir = os.path.join(list_folders[-1], f"mxf_{mxf_option}_data")

        output_spm_conv_dir = os.path.join(mxf_path_dir, "mxf_sess_data", "spm_convert_epo")
        spm_converted_file = f"spm_sub-{sub}_ses-0{ses}_task-{condition}_meg_{mxf_option}_{trigger}_nrg_ERF_epo"
        base_fname = os.path.join(output_spm_conv_dir,spm_converted_file + ".mat")

        sensor_data, sensor_time, ch_names = load_meg_sensor_data(base_fname)
        st_idx=np.where((sensor_time>=time_lims[0]) & (sensor_time<=time_lims[1]))[0]
        sensor_data=sensor_data[:,st_idx,:]
        total_trials = sensor_data.shape[2]

        sensor_time=sensor_time[st_idx]
        mean_signal = np.mean(sensor_data, axis=2)
        noise = sensor_data - mean_signal[:, :, np.newaxis]
        rms_signal = np.sqrt(np.mean(mean_signal**2, axis=0))  # shape: (n_times,)
        rms_noise = np.sqrt(np.mean(np.mean(noise**2, axis=2), axis=0))  # shape: (n_times,)
        snr_db = 20 * np.log10(rms_signal / rms_noise)

        snrs_subj.append(snr_db)
        total_trials_sub.append(total_trials)


    snrs_all = np.array(snrs_subj)
    mean_snr_sub = np.mean(snrs_all, axis=1)
    total_trials_sub = np.array(total_trials_sub)

    snr_data[trigger] = {
    'y': np.array(mean_snr_sub),
    'x': np.array(total_trials_sub),
    }

left_idx = np.where(snr_data['left_stim']['y'] < snr_lim)[0]
right_idx = np.where(snr_data['right_stim']['y'] < snr_lim)[0]
lo_snr_subjects = [item for i, item in enumerate(subjects) if i in left_idx or i in right_idx]
hi_snr_subjects = [item for i, item in enumerate(subjects) if i not in left_idx and i not in right_idx]
print(f"LO SNR subj : {lo_snr_subjects}")
print(f" HI SNR subj :{hi_snr_subjects}")

time_lims=[- 300,300]

patch_size = ["5"]
window_size = ["25", "50"]

for win in window_size:
# RIGHT Attention loop
    for patch in patch_size: 
        
        all_lh_ra_prior_ts=[]
        all_lh_ra_F_diff=[]
        all_lh_ra_layer_F_diff=[]
        all_lh_ra_bb_boundaries=[]

        all_rh_ra_prior_ts=[]
        all_rh_ra_F_diff=[]
        all_rh_ra_layer_F_diff=[]
        all_rh_ra_bb_boundaries=[]

        for subject in hi_snr_subjects:
            sub_n=subject.split('-')[-1]
            for sess_idx, session in enumerate(sessions[subject]):
                #out_fname = os.path.join(out_dir,f'results_{subject}_{session}_right_stim_patch5.npz') 
                out_fname = os.path.join(out_dir,f'results_{subject}_{session}_right_stim_patch1_ws{win}.npz')

                if os.path.exists(out_fname):
                    subj_surf_dir=os.path.join(braindyn_files,f'{sub_n}-synth','layer_surf')
                    
                    orig_inflated=nib.load(os.path.join(subj_surf_dir, 'inflated.gii'))
                    ds_inflated=nib.load(os.path.join(subj_surf_dir, 'inflated.ds.gii'))
                    ds_pial = nib.load(os.path.join(subj_surf_dir, 'pial.ds.gii'))

                    data = np.load(out_fname, allow_pickle=True)
                    lh_prior = data['lh_prior']
                    lh_prior_ts = data['lh_prior_ts']
                    rh_prior = data['rh_prior']
                    rh_prior_ts = data['rh_prior_ts']
                    ts_time = data['ts_time']
                    lh_Fs = data['lh_Fs']
                    rh_Fs = data['rh_Fs']
                    wois = data['wois']

                    woi_time=np.mean(wois,axis=-1)
                    t_idx=np.where((woi_time>=time_lims[0]) & (woi_time<=time_lims[1]))[0]
                    woi_win=woi_time[t_idx]
                    lh_Fs_win=lh_Fs[:,t_idx]
                    rh_Fs_win=rh_Fs[:,t_idx]

                    ts_idx=np.where((ts_time>=time_lims[0]) & (ts_time<=time_lims[1]))[0]
                    lh_prior_ts=lh_prior_ts[ts_idx]
                    rh_prior_ts=rh_prior_ts[ts_idx]
                    ts_time=ts_time[ts_idx]
                    
                    m_idx=np.argmax(np.abs(lh_prior_ts))
                    if lh_prior_ts[m_idx]>0:
                        lh_prior_ts=-1*lh_prior_ts
                    m_idx=np.argmax(np.abs(rh_prior_ts))
                    if rh_prior_ts[m_idx]>0:
                        rh_prior_ts=-1*rh_prior_ts

                    # Compute the relative free energy for each layer model, compared to the worst model at each time step
                    lh_F_diff = np.zeros((lh_Fs_win.shape[0],lh_Fs_win.shape[1]))
                    for t in range(lh_Fs_win.shape[1]):
                        minF = np.min(lh_Fs_win[:,t])
                        lh_F_diff[:,t] = lh_Fs_win[:,t]-minF
                    rh_F_diff = np.zeros((rh_Fs_win.shape[0],rh_Fs_win.shape[1]))
                    for t in range(rh_Fs_win.shape[1]):
                        minF = np.min(rh_Fs_win[:,t])
                        rh_F_diff[:,t] = rh_Fs_win[:,t]-minF
                        
                    lh_surf_bb_bounds=n_layers*get_bigbrain_layer_boundaries(f'{sub_n}-synth', subj_surf_dir, subj_coord=ds_pial.darrays[0].data[lh_prior,:])
                    
                    lh_idx = split_into_6_layers(n_layers, lh_surf_bb_bounds)

                    lh_layer_F = np.vstack([ 
                        np.mean(lh_Fs_win[lh_idx[0], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[1], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[2], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[3], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[4], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[5], :], axis=0), 
                    ])

                    lh_layer_F_diff = np.zeros((lh_layer_F.shape[0],lh_layer_F.shape[1]))
                    for t in range(lh_layer_F.shape[1]):
                        minF = np.nanmin(lh_layer_F[:,t])
                        lh_layer_F_diff[:,t] = lh_layer_F[:,t]-minF
                        
                    rh_surf_bb_bounds=n_layers*get_bigbrain_layer_boundaries(f'{sub_n}-synth', subj_surf_dir, subj_coord=ds_pial.darrays[0].data[rh_prior,:])
                    
                    rh_idx = split_into_6_layers(n_layers, rh_surf_bb_bounds)


                    rh_layer_F = np.vstack([
                        np.mean(rh_Fs_win[rh_idx[0], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[1], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[2], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[3], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[4], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[5], :], axis=0), 
                    ])
                    
                    rh_layer_F_diff = np.zeros((rh_layer_F.shape[0],rh_layer_F.shape[1]))
                    for t in range(rh_layer_F.shape[1]):
                        minF = np.nanmin(rh_layer_F[:,t])
                        rh_layer_F_diff[:,t] = rh_layer_F[:,t]-minF

                    all_lh_ra_prior_ts.append(lh_prior_ts)
                    all_lh_ra_F_diff.append(lh_F_diff)
                    all_lh_ra_layer_F_diff.append(lh_layer_F_diff)
                    all_lh_ra_bb_boundaries.append(lh_surf_bb_bounds)
                    
                    all_rh_ra_prior_ts.append(rh_prior_ts)
                    all_rh_ra_F_diff.append(rh_F_diff)
                    all_rh_ra_layer_F_diff.append(rh_layer_F_diff)
                    all_rh_ra_bb_boundaries.append(rh_surf_bb_bounds)
                    
        # all_snrs=np.array(all_snrs)
        all_lh_ra_prior_ts=np.array(all_lh_ra_prior_ts)
        all_lh_ra_F_diff=np.array(all_lh_ra_F_diff)
        all_lh_ra_layer_F_diff=np.array(all_lh_ra_layer_F_diff)
        all_lh_ra_bb_boundaries=np.array(all_lh_ra_bb_boundaries)

        all_rh_ra_prior_ts=np.array(all_rh_ra_prior_ts)
        all_rh_ra_F_diff=np.array(all_rh_ra_F_diff)
        all_rh_ra_layer_F_diff=np.array(all_rh_ra_layer_F_diff)
        all_rh_ra_bb_boundaries=np.array(all_rh_ra_bb_boundaries)


        # LEFT Attention Loop

        all_lh_la_prior_ts=[]
        all_lh_la_F_diff=[]
        all_lh_la_layer_F_diff=[]
        all_lh_la_bb_boundaries=[]

        all_rh_la_prior_ts=[]
        all_rh_la_F_diff=[]
        all_rh_la_layer_F_diff=[]
        all_rh_la_bb_boundaries=[]

        for subject in hi_snr_subjects:
            sub_n=subject.split('-')[-1]
            for sess_idx, session in enumerate(sessions[subject]):
                #out_fname = os.path.join(out_dir,f'results_{subject}_{session}_left_stim_patch5.npz') 
                out_fname = os.path.join(out_dir,f'results_{subject}_{session}_left_stim_patch1_ws{win}.npz')

                if os.path.exists(out_fname):
                    subj_surf_dir=os.path.join(braindyn_files,f'{sub_n}-synth','layer_surf')
                    
                    orig_inflated=nib.load(os.path.join(subj_surf_dir, 'inflated.gii'))
                    ds_inflated=nib.load(os.path.join(subj_surf_dir, 'inflated.ds.gii'))
                    ds_pial = nib.load(os.path.join(subj_surf_dir, 'pial.ds.gii'))

                    data = np.load(out_fname, allow_pickle=True)
                    lh_prior = data['lh_prior']
                    lh_prior_ts = data['lh_prior_ts']
                    rh_prior = data['rh_prior']
                    rh_prior_ts = data['rh_prior_ts']
                    ts_time = data['ts_time']
                    lh_Fs = data['lh_Fs']
                    rh_Fs = data['rh_Fs']
                    wois = data['wois']

                    woi_time=np.mean(wois,axis=-1)
                    t_idx=np.where((woi_time>=time_lims[0]) & (woi_time<=time_lims[1]))[0]
                    woi_win=woi_time[t_idx]
                    lh_Fs_win=lh_Fs[:,t_idx]
                    rh_Fs_win=rh_Fs[:,t_idx]

                    ts_idx=np.where((ts_time>=time_lims[0]) & (ts_time<=time_lims[1]))[0]
                    lh_prior_ts=lh_prior_ts[ts_idx]
                    rh_prior_ts=rh_prior_ts[ts_idx]
                    ts_time=ts_time[ts_idx]
                    
                    m_idx=np.argmax(np.abs(lh_prior_ts))
                    if lh_prior_ts[m_idx]>0:
                        lh_prior_ts=-1*lh_prior_ts
                    m_idx=np.argmax(np.abs(rh_prior_ts))
                    if rh_prior_ts[m_idx]>0:
                        rh_prior_ts=-1*rh_prior_ts

                    # Compute the relative free energy for each layer model, compared to the worst model at each time step
                    lh_F_diff = np.zeros((lh_Fs_win.shape[0],lh_Fs_win.shape[1]))
                    for t in range(lh_Fs_win.shape[1]):
                        minF = np.min(lh_Fs_win[:,t])
                        lh_F_diff[:,t] = lh_Fs_win[:,t]-minF
                    rh_F_diff = np.zeros((rh_Fs_win.shape[0],rh_Fs_win.shape[1]))
                    for t in range(rh_Fs_win.shape[1]):
                        minF = np.min(rh_Fs_win[:,t])
                        rh_F_diff[:,t] = rh_Fs_win[:,t]-minF
                        
                    lh_surf_bb_bounds=n_layers*get_bigbrain_layer_boundaries(f'{sub_n}-synth', subj_surf_dir, subj_coord=ds_pial.darrays[0].data[lh_prior,:])
                    
                    lh_idx = split_into_6_layers(n_layers, lh_surf_bb_bounds)

                    lh_layer_F = np.vstack([ 
                        np.mean(lh_Fs_win[lh_idx[0], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[1], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[2], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[3], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[4], :], axis=0), 
                        np.mean(lh_Fs_win[lh_idx[5], :], axis=0), 
                    ])
                    
                    lh_layer_F_diff = np.zeros((lh_layer_F.shape[0],lh_layer_F.shape[1]))
                    for t in range(lh_layer_F.shape[1]):
                        minF = np.nanmin(lh_layer_F[:,t])
                        lh_layer_F_diff[:,t] = lh_layer_F[:,t]-minF
                        
                    
                    rh_surf_bb_bounds=n_layers*get_bigbrain_layer_boundaries(f'{sub_n}-synth', subj_surf_dir, subj_coord=ds_pial.darrays[0].data[rh_prior,:])
                    
                    rh_idx = split_into_6_layers(n_layers, rh_surf_bb_bounds)

                    rh_layer_F = np.vstack([ 
                        np.mean(rh_Fs_win[rh_idx[0], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[1], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[2], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[3], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[4], :], axis=0), 
                        np.mean(rh_Fs_win[rh_idx[5], :], axis=0), 
                    ])
                    
                    rh_layer_F_diff = np.zeros((rh_layer_F.shape[0],rh_layer_F.shape[1]))
                    for t in range(rh_layer_F.shape[1]):
                        minF = np.nanmin(rh_layer_F[:,t])
                        rh_layer_F_diff[:,t] = rh_layer_F[:,t]-minF
                    
                    all_lh_la_prior_ts.append(lh_prior_ts)
                    all_lh_la_F_diff.append(lh_F_diff)
                    all_lh_la_layer_F_diff.append(lh_layer_F_diff)
                    all_lh_la_bb_boundaries.append(lh_surf_bb_bounds)
                    
                    all_rh_la_prior_ts.append(rh_prior_ts)
                    all_rh_la_F_diff.append(rh_F_diff)
                    all_rh_la_layer_F_diff.append(rh_layer_F_diff)
                    all_rh_la_bb_boundaries.append(rh_surf_bb_bounds)

        all_lh_la_prior_ts=np.array(all_lh_la_prior_ts)
        all_lh_la_F_diff=np.array(all_lh_la_F_diff)
        all_lh_la_layer_F_diff=np.array(all_lh_la_layer_F_diff)
        all_lh_la_bb_boundaries=np.array(all_lh_la_bb_boundaries)

        all_rh_la_prior_ts=np.array(all_rh_la_prior_ts)
        all_rh_la_F_diff=np.array(all_rh_la_F_diff)
        all_rh_la_layer_F_diff=np.array(all_rh_la_layer_F_diff)
        all_rh_la_bb_boundaries=np.array(all_rh_la_bb_boundaries)

        # IPSILATERAL (LH-LA, RH-RA) ERF peak alignment

        ipsilateral_prior_ts = np.concatenate((all_lh_la_prior_ts, all_rh_ra_prior_ts), axis = 0)
        ipsilateral_F_diff = np.concatenate((all_lh_la_F_diff,all_rh_ra_F_diff), axis = 0)
        ipsilateral_layer_F_diff = np.concatenate((all_lh_la_layer_F_diff, all_rh_ra_layer_F_diff), axis = 0)
        ipsilateral_bb_boundaries = np.concatenate((all_lh_la_bb_boundaries, all_rh_ra_bb_boundaries), axis = 0)

        plt.figure()
        plt.plot(ts_time, minmax_unit(ipsilateral_prior_ts[0]))
        lh_lags, lh_align_idx = align_to_abs_peak_with_edges_time(ipsilateral_prior_ts, peak_window=(0.05, 0.150))

        aligned_ipsilateral_prior_ts = []
        aligned_ipsilateral_F_diff = []
        aligned_ipsilateral_layer_F_diff = []

        for i in range(ipsilateral_prior_ts.shape[0]):
            idx = lh_align_idx[i]
            if i > 0:
                plt.plot(ts_time, minmax_unit(ipsilateral_prior_ts[i, idx]))
            aligned_ipsilateral_prior_ts.append(ipsilateral_prior_ts[i, idx])
            aligned_ipsilateral_F_diff.append(ipsilateral_F_diff[i, :, idx])
            aligned_ipsilateral_layer_F_diff.append(ipsilateral_layer_F_diff[i, :, idx])

        aligned_ipsilateral_prior_ts = np.asarray(aligned_ipsilateral_prior_ts)
        aligned_ipsilateral_F_diff = np.transpose(np.asarray(aligned_ipsilateral_F_diff), (0, 2, 1))
        aligned_ipsilateral_layer_F_diff = np.transpose(np.asarray(aligned_ipsilateral_layer_F_diff), (0, 2, 1))

        plt.xlabel("Time (ms)")
        plt.ylabel("Amplitude (au)")
        plt.title(f"Ipsilateral (LH-LA, RH-RA) - aligned ERFs")
        plt.tight_layout(); plt.xlim(time_lims)
        plt.show()

        # CONTRALATERAL (LH-RA, RH-LA) ERF alignment

        controlateral_prior_ts = np.concatenate((all_lh_ra_prior_ts, all_rh_la_prior_ts), axis = 0)
        controlateral_F_diff = np.concatenate((all_lh_ra_F_diff, all_rh_la_F_diff), axis = 0)
        controlateral_layer_F_diff = np.concatenate((all_lh_ra_layer_F_diff, all_rh_la_layer_F_diff), axis = 0)
        controlateral_bb_boundaries = np.concatenate((all_lh_ra_bb_boundaries, all_rh_la_bb_boundaries), axis = 0)

        plt.figure()
        plt.plot(ts_time, minmax_unit(controlateral_prior_ts[0]))
        lh_lags, lh_align_idx = align_to_abs_peak_with_edges_time(controlateral_prior_ts, peak_window=(0.05, 0.150))

        aligned_controlateral_prior_ts = []
        aligned_controlateral_F_diff = []
        aligned_controlateral_layer_F_diff = []

        for i in range(controlateral_prior_ts.shape[0]):
            idx = lh_align_idx[i]
            if i > 0:
                plt.plot(ts_time, minmax_unit(controlateral_prior_ts[i, idx]))
            aligned_controlateral_prior_ts.append(controlateral_prior_ts[i, idx])
            aligned_controlateral_F_diff.append(controlateral_F_diff[i, :, idx])
            aligned_controlateral_layer_F_diff.append(controlateral_layer_F_diff[i, :, idx])

        aligned_controlateral_prior_ts = np.asarray(aligned_controlateral_prior_ts)
        aligned_controlateral_F_diff = np.transpose(np.asarray(aligned_controlateral_F_diff), (0, 2, 1))
        aligned_controlateral_layer_F_diff = np.transpose(np.asarray(aligned_controlateral_layer_F_diff), (0, 2, 1))

        plt.xlabel("Time (ms)")
        plt.ylabel("Amplitude (au)")
        plt.title(f"Controlateral (LH-RA, RH-LH) - aligned ERFs")
        plt.tight_layout(); plt.xlim(time_lims)
        plt.show()

        # take the average
        m_aligned_controlateral_prior_ts=np.mean(aligned_controlateral_prior_ts,axis=0)
        m_aligned_controlateral_F_diff=np.mean(aligned_controlateral_F_diff,axis=0)
        m_aligned_controlateral_layer_F_diff=np.nanmean(aligned_controlateral_layer_F_diff,axis=0)
        m_controlateral_bb_boundaries=np.mean(controlateral_bb_boundaries,axis=0)

        fig, ax = plt.subplots(figsize=(12, 8))

        col_l = plt.cm.cool(np.linspace(0,1, num=6))
        l_labels=['I','II','III','IV','V','VI']
        for i in range(6):
            ax.plot(ts_time, m_aligned_controlateral_layer_F_diff[i,:],label=l_labels[i],color=col_l[i])
        ax.axvline(x=0, color='k', linestyle='--')

        plt.legend()
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel(r'$\Delta$F')
        ax2=ax.twinx()
        ax2.plot(ts_time,m_aligned_controlateral_prior_ts,'k')
        ax2.set_ylabel('Amplitude (nAm/mm^2)', rotation=90)

        ax.set_xlim([0,250])

        m_aligned_ipsilateral_prior_ts=np.mean(aligned_ipsilateral_prior_ts,axis=0)
        m_aligned_ipsilateral_F_diff=np.mean(aligned_ipsilateral_F_diff,axis=0)
        m_aligned_ipsilateral_layer_F_diff=np.nanmean(aligned_ipsilateral_layer_F_diff,axis=0)
        m_ipsilateral_bb_boundaries=np.mean(ipsilateral_bb_boundaries,axis=0)

        fig, ax = plt.subplots(figsize=(12, 8))

        col_l = plt.cm.cool(np.linspace(0,1, num=6))
        l_labels=['I','II','III','IV','V','VI']
        for i in range(6):
            ax.plot(ts_time, m_aligned_ipsilateral_layer_F_diff[i,:],label=l_labels[i],color=col_l[i])
        ax.axvline(x=0, color='k', linestyle='--')

        plt.legend()
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel(r'$\Delta$F')
        ax2=ax.twinx()
        ax2.plot(ts_time,m_aligned_ipsilateral_prior_ts,'k')
        ax2.set_ylabel('Amplitude (nAm/mm^2)', rotation=90)

        ax.set_xlim([0,250])

        savemat_fullpath = os.path.join(savemat_folder, f'aligned_V1_ERF_DeltaFs_win{win}_ps{patch}.mat')
        savemat(savemat_fullpath, {

            'ipsilateral_prior_ts':ipsilateral_prior_ts, 
            'ipsilateral_F_diff':ipsilateral_F_diff,
            'ipsilateral_layer_F_diff':ipsilateral_layer_F_diff,

            'aligned_ipsilateral_prior_ts':aligned_ipsilateral_prior_ts,
            'aligned_ipsilateral_F_diff': aligned_ipsilateral_F_diff, 
            'aligned_ipsilateral_layer_F_diff': aligned_ipsilateral_layer_F_diff,
            
            'controlateral_layer_F_diff': controlateral_layer_F_diff,
            'controlateral_F_diff': controlateral_F_diff,
            'controlateral_prior_ts': controlateral_prior_ts,

            'aligned_controlateral_prior_ts': aligned_controlateral_prior_ts,
            'aligned_controlateral_F_diff': aligned_controlateral_F_diff,
            'aligned_controlateral_layer_F_diff': aligned_controlateral_layer_F_diff, 

            'm_aligned_controlateral_prior_ts': m_aligned_controlateral_prior_ts,
            'm_aligned_controlateral_F_diff': m_aligned_controlateral_F_diff,
            'm_aligned_controlateral_layer_F_diff': m_aligned_controlateral_layer_F_diff,

            'm_aligned_ipsilateral_prior_ts': m_aligned_ipsilateral_prior_ts,
            'm_aligned_ipsilateral_F_diff': m_aligned_ipsilateral_F_diff,
            'm_aligned_ipsilateral_layer_F_diff': m_aligned_ipsilateral_layer_F_diff,

            'wois' : wois, 
            'ts_time' : ts_time,
        })

